In [43]:
import pandas as pd
df = pd.read_csv('retail_sales_dataset.csv')
df['Date'] = pd.to_datetime(df['Date'])

In [44]:
df = df.sort_values(by='Date')

In [45]:
quantity_by_category = df.groupby('Product Category') ['Quantity'].sum()

In [46]:
age_by_category = df.groupby ('Product Category') ['Age'].mean()

In [47]:
import sqlite3 
conn = sqlite3.connect('retail_sales.db')
df.to_sql('sales_data', conn, if_exists='replace', index=False)

1000

In [48]:
query = "SELECT * FROM sales_data LIMIT 5"
result = pd.read_sql(query, conn)
result

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,522,2023-01-01 00:00:00,CUST522,Male,46,Beauty,3,500,1500.0
1,180,2023-01-01 00:00:00,CUST180,Male,41,Clothing,3,300,900.0
2,559,2023-01-01 00:00:00,CUST559,Female,40,Clothing,4,300,1200.0
3,303,2023-01-02 00:00:00,CUST303,Male,19,Electronics,3,30,90.0
4,979,2023-01-02 00:00:00,CUST979,Female,19,Beauty,1,25,25.0


In [49]:
query = """
SELECT "Product Category", SUM("Total Amount") as total_revenue
FROM sales_data
GROUP BY  "Product Category"
ORDER BY total_revenue DESC
"""
reult = pd.read_sql(query, conn)
result

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,522,2023-01-01 00:00:00,CUST522,Male,46,Beauty,3,500,1500.0
1,180,2023-01-01 00:00:00,CUST180,Male,41,Clothing,3,300,900.0
2,559,2023-01-01 00:00:00,CUST559,Female,40,Clothing,4,300,1200.0
3,303,2023-01-02 00:00:00,CUST303,Male,19,Electronics,3,30,90.0
4,979,2023-01-02 00:00:00,CUST979,Female,19,Beauty,1,25,25.0


In [50]:
query = """
SELECT "Product Category", SUM("Total Amount") as total_revenue
FROM sales_data
WHERE Quantity > 2
GROUP BY "Product Category"
ORDER BY total_revenue DESC
"""
result = pd.read_sql(query, conn)
result

,Product Category,total_revenue
0,Clothing,114795.0
1,Electronics,113495.0
2,Beauty,101855.0


In [51]:
query = """
SELECT "GENDER", AVG("Total Amount") as total_amount
FROM sales_data
GROUP BY "GENDER"
"""
result = pd.read_sql(query, conn)
result

,Gender,total_amount
0,Female,457.249509
1,Male,455.428571


In [52]:
query = """
SELECT "Product Category", count("Total Amount") as number_of_transactions
FROM sales_data
GROUP BY "Product Category"
ORDER BY number_of_transactions DESC
"""
result = pd.read_sql(query, conn)
result

,Product Category,number_of_transactions
0,Clothing,351
1,Electronics,341
2,Beauty,307


In [53]:
customers = df[['Customer ID', 'Gender', 'Age']].drop_duplicates()
customers.to_sql('customers', conn, if_exists='replace', index=False)

1000

In [54]:
query = """
SELECT s."Customer ID", c."Gender", s."Total Amount"
FROM sales_data s
JOIN customers c ON s."Customer ID" = c."Customer ID"
LIMIT 5
 """
result = pd.read_sql(query, conn)
result

,Customer ID,Gender,Total Amount
0,CUST522,Male,1500.0
1,CUST180,Male,900.0
2,CUST559,Female,1200.0
3,CUST303,Male,90.0
4,CUST979,Female,25.0


In [55]:
query = """
SELECT s."Customer ID", SUM(s."Total Amount") as total_spent
FROM sales_data s
GROUP BY s."Customer ID"
ORDER BY total_spent DESC
LIMIT 1
"""
result = pd.read_sql(query, conn)
result

,Customer ID,total_spent
0,CUST970,2000.0


In [56]:
df['Age Group'] = pd.cut(df['Age'], bins=[18, 25, 35, 45, 55, 65], labels=['18-25', '26-35', '36-45', '46-55', '56-65'])
df.to_sql('sales_data', conn, if_exists='replace', index=False)

query = """
SELECT Gender, "Age Group", AVG("Total Amount") as avg_spend
FROM sales_data
GROUP BY Gender, "Age Group"
ORDER BY avg_spend DESC
"""
result = pd.read_sql(query, conn)
result

,Gender,Age Group,avg_spend
0,Female,NaN,610.769231
1,Male,18-25,522.562500
2,Female,26-35,515.093458
3,Male,36-45,475.105263
4,Female,18-25,463.676471
5,Female,46-55,450.932203
6,Male,26-35,442.500000
7,Female,36-45,436.775701
8,Male,46-55,430.727273
9,Male,56-65,426.262626


In [57]:
query = """
SELECT "Product Category", COUNT(*) as num_purchases
FROM sales_data
WHERE Age < 30
GROUP BY "Product Category"
ORDER BY num_purchases DESC
LIMIT 1
"""
result = pd.read_sql(query, conn)
result

,Product Category,num_purchases
0,Clothing,85


In [58]:
query = """
SELECT "Customer ID", COUNT(DISTINCT Age) as age_versions
FROM customers
GROUP BY "Customer ID"
HAVING age_versions > 1
"""
result = pd.read_sql(query, conn)
result

,Customer ID,age_versions


## Findings
- The highest-spending customer (CUST970) spent $2,000 total across their transactions.
- Among clearly-grouped customers, Female 18-25 had the highest average spend ($522.56), 
  and Female 56-65 had the lowest ($398.02) — consistent with the age trend found in Project 1.
- Clothing was the most purchased category among customers under 30 (85 transactions).
- No customers had conflicting age records across duplicate entries, suggesting clean customer data.

*Note: a small number of customers exactly 18 years old were excluded from age-group 
comparisons due to how the age bins were defined — a good example of why boundary 
conditions matter when bucketing continuous data.*